# Entrapment module — main plots for all public data points

Fetches every public datapoint JSON from the `Results_entrapment_ion_DIA_Astral` results repository and reproduces, in **Plotly**, the two main plots the webinterface shows for the entrapment module (`proteobench/plotting/plot_generator_entrapment.py`):

1. **Main metric scatter** — estimated FDP bound (paired method) vs. number of identified features, one point per submitted workflow, colored by software and shaped by validity category.
2. **Forest plot** — per-workflow interval from the estimated lower to upper (paired) FDP bound, with the declared FDR threshold marked, plus a side panel of identified-feature counts.

This is independent of the app's plotting code (no imports from `proteobench.plotting`) — it re-derives the same visual encoding directly from the raw datapoint fields.

**Styling** matches the other ProteoBench manuscript notebooks (`Astral_in_depth_figures.ipynb`, `Astral_diaPASEF_main_figure.ipynb`, `mean_vs_median_epsilon.ipynb`): a white plot background, Arial font, black axis lines with light gridlines, a colorblind-safe categorical palette for software identity and a fixed status palette for the validity category (both validated with the `dataviz` skill's palette checker; category is never color-alone, since marker shape/legend text always carries it too), thousands-separated axis ticks, panel labels (`panel_label="A"`/`"B"`), and PNG (3x scale) + SVG export.

In [463]:
import io
import json
import os
import tarfile

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import requests

# Shared Plotly styling, matching the manuscript figures in Astral_in_depth_figures.ipynb /
# Astral_diaPASEF_main_figure.ipynb / mean_vs_median_epsilon.ipynb: white background, Arial
# font, black axis lines with light gridlines, and 3x-scaled raster export alongside vector SVG.
_BASE_LAYOUT = dict(
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial, sans-serif", size=17, color="black"),
)
_AXIS_STYLE = dict(
    showline=True,
    linecolor="black",
    linewidth=1,
    mirror=True,
    gridcolor="lightgray",
    gridwidth=1,
    title_font=dict(size=20),
    tickfont=dict(size=16),
)
_LEGEND_STYLE = dict(
    bgcolor="rgba(255,255,255,0.8)",
    bordercolor="lightgray",
    borderwidth=1,
    font=dict(size=15),
)


def _save_figure(fig, path_without_ext):
    """PNG (3x scale) + vector SVG, matching the export convention of the other manuscript notebooks."""
    fig.write_image(f"{path_without_ext}.png", scale=3)
    fig.write_image(f"{path_without_ext}.svg")

## Configuration

`RESULTS_REPO` is the public, read-only results repository for the entrapment module (`module_id = "entrapment_DIA_ion_Astral"`). GitHub API requests are unauthenticated by default (rate-limited to 60/hour per IP) — set the `GITHUB_TOKEN` environment variable for a higher limit; never hardcode a token here.

In [464]:
GITHUB_ORG = "Proteobench"
RESULTS_REPO = "Results_entrapment_ion_DIA_Astral"
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")  # optional, avoids low unauthenticated rate limits

# Categorical palette for software identity, in a fixed assignment order (never re-cycled
# per-plot). Validated colorblind-safe for scatter/all-pairs comparison up to 8 simultaneous
# tools (worst-case CVD deltaE 9.2, normal-vision deltaE 24.0 — see dataviz skill's
# validate_palette.js). Beyond 8 simultaneous tools in one plot, re-run the validator or
# fold the long tail into "Other" — color alone is no longer guaranteed distinguishable.
SOFTWARE_COLORS = {
            "MaxQuant": "#88ccef",
            "AlphaPept": "#cc6777",
            "ProlineStudio": "#ddcc77",
            "MSAngel": "#147733",
            "FragPipe": "#342288",
            "i2MassChroQ": "#aa4599",
            "Sage": "#671100",
            "WOMBAT": "#44aa9a",
            "DIA-NN": "#999934",
            "AlphaDIA": "#1D2732",
            "Custom": "#000000",
            "Spectronaut": "#007548",
            "FragPipe (DIA-NN quant)": "#F89008",
            "MSAID": "#bfef45",
            "MetaMorpheus": "#637C7A",
            "Proteome Discoverer": "#911eb4",
            "PEAKS": "#f032e6",
            "quantms": "#f5e830"
}

# Validity category: upper FDP bound vs. declared FDR (see EntrapmentScores.categorise_metric).
# Fixed reserved status palette (not re-themed; contrast/CVD pre-validated per-role, not as a
# categorical set — see dataviz skill's palette.md). Never carries meaning by hue alone: every
# use below pairs it with a distinct marker shape (scatter) or legend text (forest bars).
CATEGORY_COLORS = {"valid": "#0ca30c", "inconclusive": "#fab219", "invalid": "#d03b3b"}
# Plotly marker symbol names (not matplotlib's single-character marker codes).
CATEGORY_MARKERS = {"valid": "circle", "inconclusive": "triangle-up", "invalid": "x"}

## Fetch all public datapoints

Each results repository stores one JSON file per submitted workflow. Downloading the repo tarball and parsing every JSON file avoids the GitHub API's per-file rate limit. Each JSON's top-level fields already include the metrics we need (`nr_id_features`, `lower_bound_FDP`, `combined_FDP`, `paired_FDP`, `category_combined`, `category_paired`, `reported_fdr_parsed_from_input`) — see `EntrapmentDatapoint`/`EntrapmentScores.calculate_metrics()`.

In [465]:
def fetch_entrapment_datapoints(repo_name=RESULTS_REPO, org=GITHUB_ORG, token=GITHUB_TOKEN):
    """Download the results repo tarball and return one row per submitted workflow."""
    headers = {"Authorization": f"token {token}"} if token else {}
    url = f"https://api.github.com/repos/{org}/{repo_name}/tarball/main"
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()

    records = []
    with tarfile.open(fileobj=io.BytesIO(resp.content), mode="r:gz") as tar:
        for member in tar.getmembers():
            if member.isfile() and member.name.endswith(".json"):
                extracted = tar.extractfile(member)
                try:
                    records.append(json.loads(extracted.read()))
                except json.JSONDecodeError:
                    continue
    return pd.DataFrame(records)

In [466]:
df = fetch_entrapment_datapoints()
print(f"{len(df)} public entrapment datapoints")
df[
    [
        "id",
        "software_name",
        "software_version",
        "nr_id_features",
        "lower_bound_FDP",
        "paired_FDP",
        "reported_fdr_parsed_from_input",
        "category_paired",
    ]
]

13 public entrapment datapoints


,id,software_name,software_version,nr_id_features,lower_bound_FDP,paired_FDP,reported_fdr_parsed_from_input,category_paired
0,DIA-NN_20260805_125948,DIA-NN,2.5.0 Academia,77926,0.002233,0.004248,0.049738,valid
1,Spectronaut_20260804_120117,Spectronaut,21.0.260602.94842,56807,0.004753,0.008573,0.009998,valid
2,DIA-NN_20260730_143657,DIA-NN,2.3.2 Academia,87473,0.004984,0.009294,0.009996,valid
3,DIA-NN_20260805_130208,DIA-NN,1.8.1,73573,0.003860,0.007367,0.009983,valid
4,DIA-NN_20260727_113032,DIA-NN,2.3.2 Academia,85980,0.005141,0.009432,0.009995,valid
5,DIA-NN_20260727_113226,DIA-NN,2.5.1 Academia,87847,0.007695,0.014366,0.049888,valid
6,DIA-NN_20260730_143818,DIA-NN,2.5.1 Academia,90421,0.007852,0.014676,0.049899,valid
7,DIA-NN_20260730_143438,DIA-NN,2.6.0 Academia,89733,0.007611,0.014343,0.049941,valid
8,DIA-NN_20260727_113552,DIA-NN,2.6.0 Academia,87172,0.007847,0.014638,0.049974,valid
9,DIA-NN_20260805_134418,DIA-NN,2.5.0 Academia,79714,0.003688,0.006900,0.049650,valid


## Filter by reported FDR threshold

Each datapoint also stores `fdp_curve` — the same estimated FDP bounds recomputed at a series of Q-value thresholds (see `EntrapmentScores.calculate_fdp_at_fdr_thresholds`). Passing `threshold=<value>` to either plot below re-derives `lower_bound_FDP`/`paired_FDP`/`nr_id_features`/`category_paired` from that curve at the given Q-value instead of each workflow's own maximum reported value — this is the same `threshold` control the webinterface exposes (`EntrapmentPlotGenerator.plot_main_metric`/`plot_forest`). It lets you compare workflows at one common FDR cutoff instead of at whatever cutoff each one happened to report. Matching is done within 1% relative tolerance of the requested value (same as the app); workflows with no curve entry near that threshold are dropped from the plot.

In [467]:
def _get_fdp_entry_at_threshold(fdp_curve, threshold):
    """Return the fdp_curve entry closest to `threshold` (within 1% relative tolerance), or {} if none."""
    if not isinstance(fdp_curve, dict) or not fdp_curve:
        return {}
    float_keys = {float(k): v for k, v in fdp_curve.items()}
    tol = threshold * 0.01
    candidates = {k: v for k, v in float_keys.items() if abs(k - threshold) <= tol}
    if not candidates:
        return {}
    best = min(candidates, key=lambda k: abs(k - threshold))
    return candidates[best]


def apply_threshold(df, threshold):
    """Re-derive FDP metrics at a fixed Q-value threshold from each row's `fdp_curve`.

    Rows without a curve entry near `threshold` are dropped. Returns `df` unchanged
    when `threshold` is None (the "maximum reported" mode used by default).
    """
    if threshold is None:
        return df

    entries = df["fdp_curve"].apply(lambda c: _get_fdp_entry_at_threshold(c, threshold))
    matched = entries.apply(bool)
    dropped = (~matched).sum()
    if dropped:
        print(f"threshold={threshold}: dropping {dropped}/{len(df)} workflow(s) with no curve entry near this value")

    out = df[matched].copy()
    entries = entries[matched]
    for col in ("lower_bound_FDP", "combined_FDP", "paired_FDP", "nr_id_features", "category_combined", "category_paired"):
        out[col] = entries.apply(lambda e: e.get(col))
    return out.reset_index(drop=True)

## Plot 1: main metric scatter

One point per workflow: estimated upper FDP bound (paired method) on x, number of identified features on y. Point color encodes the software tool (native Plotly legend); marker shape encodes the validity category (paired) — circle = valid, triangle = inconclusive, cross = invalid — shown in a second, independent legend (`legend2`), the same dual-legend pattern used for the two encoded dimensions in `mean_vs_median_epsilon.ipynb`'s overview plot. Matches `EntrapmentPlotGenerator.plot_main_metric()`'s default (`metric="Estimated upper FDP bound - Paired method"`, `colorblind_mode=False`).

In [468]:
def plot_main_metric(
    df,
    metric_col="paired_FDP",
    metric_label="Estimated upper FDP bound",
    threshold=None,
    panel_label=None,
):
    plot_df = apply_threshold(df, threshold)

    fig = go.Figure()
    seen_software = set()
    for software, sub in plot_df.groupby("software_name"):
        color = SOFTWARE_COLORS.get(software, "#000000")
        for category, cat_sub in sub.groupby("category_paired"):
            fig.add_trace(
                go.Scatter(
                    x=cat_sub[metric_col],
                    y=cat_sub["nr_id_features"],
                    mode="markers",
                    marker=dict(
                        color=color,
                        symbol=CATEGORY_MARKERS.get(category, "circle"),
                        size=12,
                        line=dict(color="black", width=0.8),
                    ),
                    name=software,
                    legendgroup=software,
                    showlegend=software not in seen_software,
                    hovertemplate=f"{software} ({category})<br>x: %{{x:.4f}}<br>y: %{{y:,}}<extra></extra>",
                )
            )
            seen_software.add(software)

    # Second, independent legend for marker shape -> validity category (paired) -- same
    # dual-legend pattern used in mean_vs_median_epsilon.ipynb's overview plot.
    for category, symbol in CATEGORY_MARKERS.items():
        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                marker=dict(color="#444444", size=12, symbol=symbol, line=dict(color="black", width=0.8)),
                name=category.capitalize(),
                legend="legend2",
            )
        )

    if threshold is not None:
        fig.add_vline(x=threshold, line_dash="dash", line_color="#898781", line_width=3)
        x_title = f"{metric_label} (Q ≤ {threshold})"
    else:
        x_title = metric_label

    fig.update_xaxes(title_text=x_title, **_AXIS_STYLE)
    fig.update_yaxes(title_text="Number of identified features", tickformat=",", **_AXIS_STYLE)
    fig.update_layout(
        **_BASE_LAYOUT,
        template="plotly_white",
        width=950,
        height=680,
        legend=dict(title="Software", x=1.02, y=1, yanchor="top", **_LEGEND_STYLE),
        legend2=dict(title="Category (paired)", x=1.02, y=0.32, yanchor="top", **_LEGEND_STYLE),
        margin=dict(l=80, r=210, t=40, b=70),
    )

    if panel_label:
        fig.add_annotation(
            text=f"<b>{panel_label}</b>",
            xref="x domain",
            yref="y domain",
            x=0,
            y=1,
            xanchor="right",
            yanchor="bottom",
            showarrow=False,
            font=dict(size=30, color="black"),
        )

    return fig

In [469]:
fig_scatter = plot_main_metric(df, panel_label="A")
_save_figure(fig_scatter, "entrapment_main_metric")
fig_scatter.show()

/tmp/ipykernel_2234263/3161180114.py:39: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


/tmp/ipykernel_2234263/3161180114.py:40: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




Same plot, filtered to a common Q-value threshold (`threshold=0.01`) instead of each workflow's own maximum reported FDR — this puts every tool on equal footing:

In [470]:
fig_scatter_thr = plot_main_metric(df, threshold=0.01)
fig_scatter_thr.show()

## Plot 2: forest plot

One row per **datapoint** — every submitted workflow gets its own row now (no longer collapsed to a single representative row per tool), sorted by number of identified features (ascending, matching the app default). Built as a two-column Plotly figure (`make_subplots`, shared y-axis). The left panel draws the estimated lower-to-upper (paired) FDP interval as a **range plot**: a thin connecting line per workflow with an explicit tick-mark endpoint at the lower and upper bound, so the minimum and maximum read clearly rather than as a featureless solid box — colored by validity category. A diamond marker marks the declared FDR (`reported_fdr_parsed_from_input`). The right-hand panel shows the identified-feature count per workflow, matching `EntrapmentPlotGenerator.plot_forest()`.

The y-axis tick for each row shows its `search_engine` (falls back to `software_name` if that column is absent). Rows are placed at plain numeric positions rather than on a categorical axis keyed by that text — necessary because several rows can share the same base name (e.g. 8 DIA-NN submissions), and a categorical axis would silently collapse same-text rows onto a single tick instead of giving each its own row.

Because most rows would otherwise read as an undifferentiated block of `"DIA-NN"` ticks, `plot_forest()` takes an optional `labels` argument — a dict keyed by workflow `id` — that shows an annotation on a second line underneath that row's tick (via Plotly's `<br>` tick-text tag), e.g. `"DIA-NN"` becomes two lines, `"DIA-NN"` / `"label"`, with no brackets, so individual datapoints stay distinguishable. Fill in `WORKFLOW_LABELS` below to annotate specific workflows; it's empty (no annotations) by default.

In [471]:
def plot_forest(df, sort_ascending=True, threshold=None, panel_label=None, y_label_col="search_engine", labels=None):
    plot_df = apply_threshold(df, threshold)
    plot_df = plot_df.sort_values("nr_id_features", ascending=sort_ascending).reset_index(drop=True)
    n = len(plot_df)
    labels = labels or {}

    # Row label text: base name (search_engine, falling back to software_name/id) plus an
    # optional annotation from `labels` (keyed by workflow id) on a second line underneath it
    # (via "<br>", no brackets) to disambiguate datapoints that share the same base name (e.g.
    # several DIA-NN submissions). Rendered as centered annotations below (not axis tick text,
    # which has no per-line alignment control), so a shorter label line centers under a longer
    # tool-name line instead of sharing its right edge.
    base_names = plot_df[y_label_col] if y_label_col in plot_df.columns else plot_df["id"]
    y_labels = [
        f"{base}<br>{labels[wid]}" if wid in labels else base for base, wid in zip(base_names, plot_df["id"])
    ]
    # Numeric row positions, NOT the tick text -- every datapoint gets its own row this way,
    # even when multiple rows share the same y_labels text (a categorical y-axis would instead
    # collapse same-text rows onto a single tick, merging their bars).
    y_pos = list(range(n))
    # Two-line ticks (a row with a label) need a bit more vertical room per row than a single
    # line does, so the tick text for adjacent rows doesn't run into each other.
    _row_height = 60 if labels else 50

    fig = make_subplots(
        rows=1,
        cols=2,
        shared_yaxes=True,
        column_widths=[0.65, 0.35],
        horizontal_spacing=0.04,
    )

    # FDP interval panel: a range plot, not a bar -- a thin connecting line from the lower to
    # the upper (paired) FDP bound per workflow, with an explicit tick-mark endpoint at each
    # bound so the minimum and maximum read clearly (mirrors the "|" end caps of the original
    # matplotlib forest plot; a plain go.Bar renders as a featureless solid box instead).
    for category, cat_sub in plot_df.groupby("category_paired"):
        color = CATEGORY_COLORS.get(category, "#999999")
        positions = cat_sub.index.tolist()  # row index == y position (plot_df was reset_index(drop=True))
        row_labels = [y_labels[i] for i in positions]
        lowers = cat_sub["lower_bound_FDP"].tolist()
        uppers = cat_sub["paired_FDP"].tolist()

        line_x, line_y = [], []
        for lo, hi, pos in zip(lowers, uppers, positions):
            line_x += [lo, hi, None]
            line_y += [pos, pos, None]
        fig.add_trace(
            go.Scatter(
                x=line_x,
                y=line_y,
                mode="lines",
                line=dict(color=color, width=5),
                name=category.capitalize(),
                legendgroup=category,
                hoverinfo="skip",
            ),
            row=1,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=lowers + uppers,
                y=positions + positions,
                mode="markers",
                marker=dict(symbol="line-ns", size=14, line=dict(width=3, color=color)),
                customdata=[[lbl, "Lower bound"] for lbl in row_labels]
                + [[lbl, "Upper bound"] for lbl in row_labels],
                legendgroup=category,
                showlegend=False,
                hovertemplate=f"%{{customdata[0]}} ({category})<br>%{{customdata[1]}}: %{{x:.4f}}<extra></extra>",
            ),
            row=1,
            col=1,
        )

    # Declared FDR marker (diamond) -- only meaningful in "maximum reported" mode
    # (threshold=None); at a fixed threshold every workflow shares the same Q-value cutoff,
    # shown as a dashed vline instead.
    if threshold is None:
        fig.add_trace(
            go.Scatter(
                x=plot_df["reported_fdr_parsed_from_input"].fillna(0.01),
                y=y_pos,
                mode="markers",
                marker=dict(symbol="diamond", color="#0b0b0b", size=10),
                name="Declared FDR",
                customdata=y_labels,
                hovertemplate="%{customdata}<br>Declared FDR: %{x:.4f}<extra></extra>",
            ),
            row=1,
            col=1,
        )
    else:
        fig.add_vline(x=threshold, line_dash="dash", line_color="#898781", line_width=3, row=1, col=1)
        fig.add_annotation(
            text=f"Q ≤ {threshold}",
            x=threshold,
            y=y_pos[-1],
            xref="x",
            yref="y",
            xanchor="left",
            yanchor="bottom",
            showarrow=False,
            font=dict(size=16, color="#52514e"),
        )

    fig.add_trace(
        go.Bar(
            y=y_pos,
            x=plot_df["nr_id_features"],
            orientation="h",
            marker=dict(
                color=[CATEGORY_COLORS.get(c, "#999999") for c in plot_df["category_paired"]],
                opacity=0.6,
            ),
            width=0.6,
            showlegend=False,
            customdata=y_labels,
            hovertemplate="%{customdata}<br>%{x:,} identified features<extra></extra>",
        ),
        row=1,
        col=2,
    )

    fig.update_yaxes(tickvals=y_pos, showticklabels=False, row=1, col=1, **_AXIS_STYLE)
    fig.update_yaxes(
        tickvals=y_pos,
        showticklabels=False,
        matches="y",
        row=1,
        col=2,
        **_AXIS_STYLE,
    )

    # Row labels as centered annotations rather than native tick text: annotation.align="center"
    # horizontally centers a shorter second line (the optional label) under a longer first line
    # (the tool name) -- axis tick labels have no equivalent per-line alignment control, they
    # only support flush-left/right multi-line text.
    _tick_font_size = _AXIS_STYLE["tickfont"]["size"]
    for pos, lbl in zip(y_pos, y_labels):
        fig.add_annotation(
            text=lbl,
            xref="x domain",
            yref="y",
            x=0,
            y=pos,
            xanchor="right",
            yanchor="middle",
            align="center",
            showarrow=False,
            font=dict(size=_tick_font_size, color="black"),
        )

    fig.update_xaxes(
        title=dict(text="Estimated FDP: lower bound to upper bound", standoff=8),
        row=1,
        col=1,
        **_AXIS_STYLE,
    )
    fig.update_xaxes(
        title=dict(text="Nr. identified features", standoff=8), tickformat=",", row=1, col=2, **_AXIS_STYLE
    )

    fig.update_layout(
        **_BASE_LAYOUT,
        template="plotly_white",
        width=1150,
        height=max(340, _row_height * n + 170),
        legend=dict(title="Category (paired)", **_LEGEND_STYLE),
        margin=dict(l=190, r=40, t=40, b=70),
    )

    if panel_label:
        fig.add_annotation(
            text=f"<b>{panel_label}</b>",
            xref="x domain",
            yref="y domain",
            x=0,
            y=1,
            xanchor="right",
            yanchor="bottom",
            showarrow=False,
            font=dict(size=30, color="black"),
        )

    return fig

In [472]:
df

,old_new,id,software_name,software_version,search_engine,search_engine_version,ident_fdr_psm,ident_fdr_peptide,ident_fdr_protein,enable_match_between_runs,...,min_fragment_mz,max_fragment_mz,quantification_method,protein_inference,abundance_normalization_ions,predictors_library,scan_window,postprocessing_performed,postprocessing_description,submission_comments
0,new,DIA-NN_20260805_125948,DIA-NN,2.5.0 Academia,DIA-NN,2.5.0 Academia,0.01,nan,nan,True,...,150,2000,QuantUMS high-precision,Genes,RT-dependent normalization,DIANN,6,False,nan,--dg-keep-cterm 2 --dg-min-shuffle 2.0 --dg-mi...
1,new,Spectronaut_20260804_120117,Spectronaut,21.0.260602.94842,Spectronaut,21.0.260602.94842,0.01,nan,0.01,False,...,200,3000,MS2,IDPicker,True,nan,Dynamic,False,nan,\n\n ### Parameter changes detected:\n- **semi...
2,new,DIA-NN_20260730_143657,DIA-NN,2.3.2 Academia,DIA-NN,2.3.2 Academia,0.01,nan,nan,True,...,200,1800,QuantUMS high-precision,1,RT-dependent normalization,User defined speclib,6,False,nan,\n\n ### No parameter changes detected. \n\n\n...
3,new,DIA-NN_20260805_130208,DIA-NN,1.8.1,DIA-NN,1.8.1,0.01,nan,nan,True,...,150,2000,QuantUMS high-precision,Genes,RT-dependent normalization,DIANN,6,False,nan,\n\n ### No parameter changes detected. \n\n\n...
4,new,DIA-NN_20260727_113032,DIA-NN,2.3.2 Academia,DIA-NN,2.3.2 Academia,0.01,nan,nan,True,...,200,1800,QuantUMS high-precision,1,RT-dependent normalization,User defined speclib,6,False,nan,\n\n ### No parameter changes detected. \n\n\n...
5,new,DIA-NN_20260727_113226,DIA-NN,2.5.1 Academia,DIA-NN,2.5.1 Academia,0.01,nan,nan,True,...,200,1800,QuantUMS high-precision,1,RT-dependent normalization,User defined speclib,6,False,nan,\n\n ### No parameter changes detected. \n\n\n...
6,new,DIA-NN_20260730_143818,DIA-NN,2.5.1 Academia,DIA-NN,2.5.1 Academia,0.01,nan,nan,True,...,200,1800,QuantUMS high-precision,1,RT-dependent normalization,User defined speclib,6,False,nan,\n\n ### No parameter changes detected. \n\n\n...
7,new,DIA-NN_20260730_143438,DIA-NN,2.6.0 Academia,DIA-NN,2.6.0 Academia,0.01,nan,nan,True,...,200,1800,QuantUMS high-precision,1,RT-dependent normalization,User defined speclib,6,False,nan,\n\n ### No parameter changes detected. \n\n\n...
8,new,DIA-NN_20260727_113552,DIA-NN,2.6.0 Academia,DIA-NN,2.6.0 Academia,0.01,nan,nan,True,...,200,1800,QuantUMS high-precision,1,RT-dependent normalization,User defined speclib,6,False,nan,\n\n ### No parameter changes detected. \n\n\n...
9,new,DIA-NN_20260805_134418,DIA-NN,2.5.0 Academia,DIA-NN,2.5.0 Academia,0.01,nan,nan,True,...,150,2000,QuantUMS high-precision,Genes,RT-dependent normalization,DIANN,6,False,nan,--dg-keep-cterm 3 --dg-keep-nterm 3 --dg-min-s...


In [473]:
# Every submitted workflow now gets its own row in the forest plot (no more collapsing
# multiple submissions from the same tool into one representative row). Optional per-workflow
# labels, keyed by workflow `id`, get shown on a second line underneath that row's y-axis tick
# (e.g. "DIA-NN" becomes two lines: "DIA-NN" / "label", no brackets) so individual datapoints
# stay distinguishable when a tool has several submissions. Fill this in as needed; a workflow
# left out keeps its plain search_engine/software name with no second line.
subset_ids = ["DIA-NN_20260805_125948","Spectronaut_20260804_120117","DIA-NN_20260805_134418", "AlphaDIA_20260728_142047"]
subset_labels = {
    "DIA-NN_20260805_125948": "vD_E",
    "Spectronaut_20260804_120117": "S_E",
    "DIA-NN_20260805_134418": "iD_E",
    "AlphaDIA_20260728_142047": "A_E"
}

subset_df = df[df["id"].isin(subset_ids)]

In [474]:
fig_forest = plot_forest(subset_df, panel_label="B", labels=subset_labels)
_save_figure(fig_forest, "entrapment_forest")
fig_forest.show()

/tmp/ipykernel_2234263/3161180114.py:39: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


/tmp/ipykernel_2234263/3161180114.py:40: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




Same forest plot at the same fixed threshold — note the declared-FDR marker disappears (every workflow now shares the same Q-value cutoff, shown as the dashed vline instead):

In [475]:
fig_forest_thr = plot_forest(subset_df, threshold=0.01, labels=subset_labels)
fig_forest_thr.show()

## Combined figure

Both `threshold=0.01` plots side by side in one figure — panel A (scatter) and panel B (forest, itself two panels: FDP interval + feature count) — each labeled in its own top-left corner. Rather than drawing directly into a shared set of axes (a matplotlib idiom that doesn't translate to Plotly), each panel is first built as its own standalone Plotly figure by the same functions used above, then merged into one `make_subplots` grid by copying traces, axis styles, shapes, and annotations across — the same technique `Astral_in_depth_figures.ipynb` uses to assemble its 3x2 grid from independently-built panel figures. Native per-panel legends are disabled on the merged traces and replaced by two shared swatch legends (software, validity category) stacked below the figure.

In [476]:
# ---- Build panel A (scatter) and panel B (forest) as standalone figures, then merge ----
_fig_scatter = plot_main_metric(df, threshold=0.01, panel_label="A")
_fig_forest = plot_forest(subset_df, threshold=0.01, panel_label="B", labels=subset_labels)


def _axis_ref(n, letter):
    return letter if n == 1 else f"{letter}{n}"


def _trace_axis_num(trace, letter):
    ref = getattr(trace, f"{letter}axis", None) or letter
    return 1 if ref == letter else int(ref[1:])


def _ref_axis_num(ref):
    if not ref or ref == "paper":
        return None
    ref = ref.replace(" domain", "")
    return 1 if ref in ("x", "y") else int(ref[1:])


def _apply_axis(src, dst_key, fig):
    j = src.to_plotly_json()
    for k in ("domain", "anchor", "matches"):
        j.pop(k, None)
    if j:
        fig.update_layout(**{dst_key: j})


def _remap_ref(ref, old_n, new_n):
    if not ref:
        return ref
    old_x, old_y = _axis_ref(old_n, "x"), _axis_ref(old_n, "y")
    new_x, new_y = _axis_ref(new_n, "x"), _axis_ref(new_n, "y")
    return {
        old_x: new_x,
        f"{old_x} domain": f"{new_x} domain",
        old_y: new_y,
        f"{old_y} domain": f"{new_y} domain",
    }.get(ref, ref)


combined_fig = make_subplots(rows=1, cols=3)

# Custom column domains (override make_subplots' uniform spacing): a normal gap between the
# scatter panel and the forest panel group, but an almost-zero gap between the forest plot's
# own two columns (FDP interval / feature count) so they read as one continuous panel, the way
# they do inside plot_forest() on its own. _apply_axis (below) never touches "domain", so these
# survive the axis-style copying untouched.
combined_fig.update_xaxes(domain=[0.00, 0.38], row=1, col=1)
combined_fig.update_xaxes(domain=[0.46, 0.775], row=1, col=2)
combined_fig.update_xaxes(domain=[0.78, 1.0], row=1, col=3)

# (source figure, {source subplot number -> destination column}) -- the scatter figure has one
# subplot (1); the forest figure has two (1 = FDP interval, 2 = identified-feature count).
_panels = [(_fig_scatter, {1: 1}), (_fig_forest, {1: 2, 2: 3})]

for _fig, _col_map in _panels:
    for _tr in _fig.data:
        _old_n = _trace_axis_num(_tr, "x")
        _new_col = _col_map.get(_old_n)
        if _new_col is None:
            continue
        _tr.update(showlegend=False)  # native per-panel legends replaced by the shared swatch legends below
        combined_fig.add_trace(_tr, row=1, col=_new_col)

    for _old_n, _new_col in _col_map.items():
        _apply_axis(getattr(_fig.layout, _axis_ref(_old_n, "xaxis")), _axis_ref(_new_col, "xaxis"), combined_fig)
        _apply_axis(getattr(_fig.layout, _axis_ref(_old_n, "yaxis")), _axis_ref(_new_col, "yaxis"), combined_fig)

    for _sh in _fig.layout.shapes or []:
        _j = {k: v for k, v in _sh.to_plotly_json().items() if v is not None}
        _old_n = _ref_axis_num(_j.get("xref")) or 1
        _new_n = _col_map.get(_old_n, _old_n)
        _j["xref"] = _remap_ref(_j.get("xref"), _old_n, _new_n)
        _j["yref"] = _remap_ref(_j.get("yref"), _old_n, _new_n)
        combined_fig.add_shape(**_j)

    for _an in _fig.layout.annotations or []:
        _j = {k: v for k, v in _an.to_plotly_json().items() if v is not None}
        _old_n = _ref_axis_num(_j.get("xref")) or 1
        _new_n = _col_map.get(_old_n, _old_n)
        _j["xref"] = _remap_ref(_j.get("xref"), _old_n, _new_n)
        _j["yref"] = _remap_ref(_j.get("yref"), _old_n, _new_n)
        combined_fig.add_annotation(**_j)

In [477]:
# ---- Shared swatch legends (software, category), stacked below the figure ----
# Unicode glyphs mirroring the Plotly marker symbols used for validity category in the
# scatter panel, so the swatch legend can show a category's color (as used in the forest
# panels) and its shape (as used in the scatter panel) together, not color alone.
_MARKER_GLYPHS = {"circle": "●", "triangle-up": "▲", "x": "✕"}


def _add_swatch_legend(
    fig,
    colors,
    y,
    x_start=0.0,
    x_end=1.0,
    font_size=18,
    display_fn=lambda name: name,
    title=None,
    title_width=0.12,
    markers=None,
):
    if title:
        fig.add_annotation(
            xref="paper",
            yref="paper",
            x=x_start,
            y=y,
            text=f"<b>{title}:</b>",
            showarrow=False,
            font=dict(size=font_size),
            xanchor="left",
            yanchor="middle",
        )
        x_start = x_start + title_width

    n = len(colors)
    span = x_end - x_start
    swatch_w = 18 / 1500
    swatch_h = 18 / 650
    for i, (name, col) in enumerate(colors.items()):
        xc = x_start + span * (i + 0.5) / n
        fig.add_shape(
            type="rect",
            xref="paper",
            yref="paper",
            x0=xc - swatch_w * 2.8,
            x1=xc - swatch_w * 1.4,
            y0=y - swatch_h / 2,
            y1=y + swatch_h / 2,
            fillcolor=col,
            line_width=0,
            layer="above",
        )
        text_x = xc - swatch_w * 1.0
        if markers:
            glyph = _MARKER_GLYPHS.get(markers.get(name), "")
            if glyph:
                fig.add_annotation(
                    xref="paper",
                    yref="paper",
                    x=text_x,
                    y=y,
                    text=glyph,
                    showarrow=False,
                    font=dict(size=font_size + 2, color=col),
                    xanchor="left",
                    yanchor="middle",
                )
                text_x = text_x + swatch_w * 3.2
        fig.add_annotation(
            xref="paper",
            yref="paper",
            x=text_x,
            y=y,
            text=display_fn(name),
            showarrow=False,
            font=dict(size=font_size),
            xanchor="left",
            yanchor="middle",
        )


_tool_colors = {name: col for name, col in SOFTWARE_COLORS.items() if name in df["software_name"].unique()}
# Restricted to columns 1-2's combined width (x_end=0.78) so no swatch entry lands under the
# narrow "Nr. identified features" column, and pushed further down (y, plus the larger bottom
# margin below) so the legend rows clear that column's x-axis title -- this is what was
# overlapping ("Spectronaut" landing under "Nr. identified features") before the fix.
_add_swatch_legend(combined_fig, _tool_colors, y=-0.22, x_end=0.78, title="Software")
_add_swatch_legend(
    combined_fig,
    CATEGORY_COLORS,
    y=-0.35,
    x_end=0.78,
    title="Category (paired)",
    display_fn=lambda c: c.capitalize(),
    markers=CATEGORY_MARKERS,
)

In [478]:
# ---- Global layout, save, show ----
# Height scales with the number of forest rows -- now every datapoint gets its own row (no more
# collapsing to one row per tool), so this can be much taller than a fixed size would allow for.
_n_forest_rows = len(_fig_forest.layout.yaxis.tickvals)
_fig_height = max(680, 50 * _n_forest_rows + 510)

combined_fig.update_layout(
    **_BASE_LAYOUT,
    template="plotly_white",
    width=1500,
    height=_fig_height,
    showlegend=False,
    margin=dict(l=90, r=40, t=60, b=280),
)

_save_figure(combined_fig, "entrapment_combined_threshold")
combined_fig.show()

/tmp/ipykernel_2234263/3161180114.py:39: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


/tmp/ipykernel_2234263/3161180114.py:40: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




## Combined figure with an extra Q ≤ 0.001 row

Same composite as above, but as a full 2-row x 3-col grid instead of 1x3: the top row is panel **A** (scatter) + panel **C** (forest), both at Q ≤ 0.01, and the bottom row is panel **B** (scatter) + panel **D** (forest), the same two plots recomputed at Q ≤ 0.001. **A and B are the two scatter panels; C and D are the two forest ("range + bar") panels.** Every one of the 6 cells is now its own subplot (no `rowspan`); the middle and right (forest) columns are split by threshold exactly like the left (scatter) column, instead of spanning both rows at a single threshold. Within each row, the gap between the forest plot's own two columns (FDP interval / feature count) is set to almost nothing via explicit column domains, so they read as one continuous panel — the way they do inside `plot_forest()` on its own.

This reuses `plot_main_metric`/`plot_forest` as-is, and the same trace/axis/shape/annotation-copying helpers (`_axis_ref`, `_trace_axis_num`, `_ref_axis_num`, `_apply_axis`, `_remap_ref`) and the `_add_swatch_legend` helper defined in the "Combined figure" section above — only the destination grid, panel labels, column domains, and the source-to-destination position mapping change.

In [479]:
# ---- Build panel A (scatter, Q<=0.01), panel B (scatter, Q<=0.001), panel C (forest,
# Q<=0.01), panel D (forest, Q<=0.001) ----
_fig_scatter_a = plot_main_metric(df, threshold=0.01, panel_label="A")
_fig_scatter_b = plot_main_metric(df, threshold=0.001, panel_label="B")
_fig_forest_c = plot_forest(subset_df, threshold=0.01, panel_label="C", labels=subset_labels)
_fig_forest_d = plot_forest(subset_df, threshold=0.001, panel_label="D", labels=subset_labels)

# Destination subplot numbering for the 2x3 grid below, in the row-major order Plotly assigns
# axes to subplots when there are no rowspans: (1,1)->1, (1,2)->2, (1,3)->3, (2,1)->4, (2,2)->5,
# (2,3)->6.
_dest_axis_num = {(1, 1): 1, (1, 2): 2, (1, 3): 3, (2, 1): 4, (2, 2): 5, (2, 3): 6}

combined_fig2 = make_subplots(rows=2, cols=3, row_heights=[0.5, 0.5], vertical_spacing=0.17)

# Custom column domains (same for both rows -- only the row/y-domain differs): a normal gap
# between the scatter column and the forest column group, but an almost-zero gap between the
# forest plot's own two columns (FDP interval / feature count) so they read as one continuous
# panel, the way they do inside plot_forest() on its own.
for _row in (1, 2):
    combined_fig2.update_xaxes(domain=[0.00, 0.38], row=_row, col=1)
    combined_fig2.update_xaxes(domain=[0.46, 0.775], row=_row, col=2)
    combined_fig2.update_xaxes(domain=[0.78, 1.0], row=_row, col=3)

# (source figure, {source subplot number -> destination (row, col)}) -- reuses the
# _axis_ref/_trace_axis_num/_ref_axis_num/_apply_axis/_remap_ref helpers defined in the
# "Combined figure" section above. Row 1 = Q<=0.01 (A, C); row 2 = Q<=0.001 (B, D).
_panels2 = [
    (_fig_scatter_a, {1: (1, 1)}),
    (_fig_forest_c, {1: (1, 2), 2: (1, 3)}),
    (_fig_scatter_b, {1: (2, 1)}),
    (_fig_forest_d, {1: (2, 2), 2: (2, 3)}),
]

for _fig, _pos_map in _panels2:
    for _tr in _fig.data:
        _old_n = _trace_axis_num(_tr, "x")
        _pos = _pos_map.get(_old_n)
        if _pos is None:
            continue
        _tr.update(showlegend=False)  # native per-panel legends replaced by the shared swatch legends below
        combined_fig2.add_trace(_tr, row=_pos[0], col=_pos[1])

    for _old_n, _pos in _pos_map.items():
        _new_n = _dest_axis_num[_pos]
        _apply_axis(getattr(_fig.layout, _axis_ref(_old_n, "xaxis")), _axis_ref(_new_n, "xaxis"), combined_fig2)
        _apply_axis(getattr(_fig.layout, _axis_ref(_old_n, "yaxis")), _axis_ref(_new_n, "yaxis"), combined_fig2)

    for _sh in _fig.layout.shapes or []:
        _j = {k: v for k, v in _sh.to_plotly_json().items() if v is not None}
        _old_n = _ref_axis_num(_j.get("xref")) or 1
        _new_n = _dest_axis_num[_pos_map.get(_old_n, (1, 1))]
        _j["xref"] = _remap_ref(_j.get("xref"), _old_n, _new_n)
        _j["yref"] = _remap_ref(_j.get("yref"), _old_n, _new_n)
        combined_fig2.add_shape(**_j)

    for _an in _fig.layout.annotations or []:
        _j = {k: v for k, v in _an.to_plotly_json().items() if v is not None}
        _old_n = _ref_axis_num(_j.get("xref")) or 1
        _new_n = _dest_axis_num[_pos_map.get(_old_n, (1, 1))]
        _j["xref"] = _remap_ref(_j.get("xref"), _old_n, _new_n)
        _j["yref"] = _remap_ref(_j.get("yref"), _old_n, _new_n)
        combined_fig2.add_annotation(**_j)

In [480]:
# ---- Shared swatch legends (software, category), reusing the helper defined above ----
_tool_colors2 = {name: col for name, col in SOFTWARE_COLORS.items() if name in df["software_name"].unique()}
_add_swatch_legend(combined_fig2, _tool_colors2, y=-0.19, x_end=0.78, title="Software")
_add_swatch_legend(
    combined_fig2,
    CATEGORY_COLORS,
    y=-0.30,
    x_end=0.78,
    title="Category (paired)",
    display_fn=lambda c: c.capitalize(),
    markers=CATEGORY_MARKERS,
)

In [481]:
# ---- Global layout, save, show ----
# Height scales with the number of forest rows in each threshold's row (C and D can differ
# slightly if apply_threshold() drops a different set of workflows at 0.01 vs 0.001) -- now
# every datapoint gets its own row, so this can be much taller than a fixed size would allow.
_n_forest_rows2 = max(len(_fig_forest_c.layout.yaxis.tickvals), len(_fig_forest_d.layout.yaxis.tickvals))
_fig_height2 = max(950, 2 * (50 * _n_forest_rows2 + 170) + 330)

combined_fig2.update_layout(
    **_BASE_LAYOUT,
    template="plotly_white",
    width=1500,
    height=_fig_height2,
    showlegend=False,
    margin=dict(l=90, r=40, t=60, b=270),
)

_save_figure(combined_fig2, "entrapment_combined_threshold_with_extra_panel")
combined_fig2.show()
combined_fig2.write_image("entrapment_combined_threshold_with_extra_panel.png", scale=3)
combined_fig2.write_image("entrapment_combined_threshold_with_extra_panel.svg")

/tmp/ipykernel_2234263/3161180114.py:39: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


/tmp/ipykernel_2234263/3161180114.py:40: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




/tmp/ipykernel_2234263/107947709.py:19: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


/tmp/ipykernel_2234263/107947709.py:20: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




In [482]:
subset_df

,old_new,id,software_name,software_version,search_engine,search_engine_version,ident_fdr_psm,ident_fdr_peptide,ident_fdr_protein,enable_match_between_runs,...,min_fragment_mz,max_fragment_mz,quantification_method,protein_inference,abundance_normalization_ions,predictors_library,scan_window,postprocessing_performed,postprocessing_description,submission_comments
0,new,DIA-NN_20260805_125948,DIA-NN,2.5.0 Academia,DIA-NN,2.5.0 Academia,0.01,nan,nan,True,...,150,2000,QuantUMS high-precision,Genes,RT-dependent normalization,DIANN,6,False,nan,--dg-keep-cterm 2 --dg-min-shuffle 2.0 --dg-mi...
1,new,Spectronaut_20260804_120117,Spectronaut,21.0.260602.94842,Spectronaut,21.0.260602.94842,0.01,nan,0.01,False,...,200,3000,MS2,IDPicker,True,nan,Dynamic,False,nan,\n\n ### Parameter changes detected:\n- **semi...
9,new,DIA-NN_20260805_134418,DIA-NN,2.5.0 Academia,DIA-NN,2.5.0 Academia,0.01,nan,nan,True,...,150,2000,QuantUMS high-precision,Genes,RT-dependent normalization,DIANN,6,False,nan,--dg-keep-cterm 3 --dg-keep-nterm 3 --dg-min-s...
10,new,AlphaDIA_20260728_142047,AlphaDIA,2.1.2,AlphaDIA,nan,0.01,nan,0.01,False,...,100,1700,DirectLFQ,heuristic,DirectLFQ,AlphaPeptDeep,nan,False,nan,\n\n ### No parameter changes detected. \n\n\n...
